In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv(r"D:\VS CODE\PROJECTS\Property Price Prediction\House Price.csv")
df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

EDA & VIZUALIZATION

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['TARGET(PRICE_IN_LACS)'], kde=True)
plt.title("Price Distribution")
plt.show()

plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

sns.boxplot(x=df['SQUARE_FT'])
plt.show()

SKEWNESS

In [ ]:
print("\nSkewness:\n", df.skew(numeric_only=True))
df['SQUARE_FT'] = np.log1p(df['SQUARE_FT'])
df['TARGET(PRICE_IN_LACS)'] = np.log1p(df['TARGET(PRICE_IN_LACS)'])
print("After Transformation:\n", df.skew(numeric_only=True))

FEATURE SCALING AND ENCODING

In [ ]:
df = df[['BHK_NO.', 'BHK_OR_RK', 'SQUARE_FT',
         'READY_TO_MOVE', 'UNDER_CONSTRUCTION',
         'RERA', 'TARGET(PRICE_IN_LACS)']]

In [ ]:
le = LabelEncoder()
df['BHK_OR_RK'] = le.fit_transform(df['BHK_OR_RK'])  # BHK=0, RK=1

TRAIN AND TEST SPLIT

In [ ]:
X = df.drop('TARGET(PRICE_IN_LACS)', axis=1)
y = df['TARGET(PRICE_IN_LACS)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

SCALING

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor()
}

In [ ]:
def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    def metrics(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        return mae, mse, rmse, r2

    train_metrics = metrics(y_train, y_train_pred)
    test_metrics = metrics(y_test, y_test_pred)

    return train_metrics, test_metrics

MODEL COMPARISON TABLE

In [ ]:
results = []

for name, model in models.items():
    train_m, test_m = evaluate(model, X_train, X_test, y_train, y_test)

    results.append({
        "Model": name,
        "Train RMSE": train_m[2],
        "Test RMSE": test_m[2],
        "Train R2": train_m[3],
        "Test R2": test_m[3],
        "Overfitting": "Yes" if train_m[3] - test_m[3] > 0.1 else "No"
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="Test R2", ascending=False))

HYPERPARA METER TUNING

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best Params:", grid.best_params_)

SAVE MODEL

In [ ]:
import pickle
pickle.dump(best_model, open("newmodel.pkl", "wb"))
pickle.dump(scaler, open("newscaler.pkl", "wb"))
pickle.dump(le, open("newencoder.pkl", "wb"))

import streamlit as st
import numpy as np
import pickle


model = pickle.load(open("newmodel.pkl", "rb"))
scaler = pickle.load(open("newscaler.pkl", "rb"))
encoder = pickle.load(open("newencoder.pkl", "rb"))


st.set_page_config(page_title="House Price Predictor", page_icon="🏠", layout="centered")


st.markdown("""
    <style>
    .stApp {
        background-image: url("https://images.unsplash.com/photo-1560518883-ce09059eeffa");
        background-size: cover;
        background-position: center;
        background-attachment: fixed;
    }

    .main-box {
        background: rgba(0, 0, 0, 0.6);
        padding: 30px;
        border-radius: 15px;
        color: white;
        box-shadow: 0px 0px 20px rgba(0,0,0,0.5);
    }

    h1 {
        text-align: center;
        color: white;
    }

    .stButton>button {
        width: 100%;
        background-color: #ff4b4b;
        color: white;
        border-radius: 10px;
        height: 3em;
        font-size: 18px;
        font-weight: bold;
    }

    .stSelectbox label, .stNumberInput label {
        color: white !important;
        font-weight: bold;
    }

    </style>
""", unsafe_allow_html=True)


st.markdown("<h1>🏠 House Price Prediction</h1>", unsafe_allow_html=True)


with st.container():
    st.markdown('<div class="main-box">', unsafe_allow_html=True)

    col1, col2 = st.columns(2)

    with col1:
        bhk = st.selectbox("BHK", [1,2,3,4,5,6,7,8,9,10])
        bhk_type = st.selectbox("Type", ["BHK", "RK"])
        square_ft = st.number_input("Square Feet", 100, 50000)

    with col2:
        ready = st.selectbox("Ready to Move", ["Yes", "No"])
        under = st.selectbox("Under Construction", ["Yes", "No"])
        rera = st.selectbox("RERA Status", ["Approved", "Not Approved"])

   
    bhk_type_enc = encoder.transform([bhk_type])[0]
    ready = 1 if ready == "Yes" else 0
    under = 1 if under == "Yes" else 0
    rera = 1 if rera == "Approved" else 0

    square_ft = np.log1p(square_ft)

    features = np.array([[bhk, bhk_type_enc, square_ft, ready, under, rera]])
    features = scaler.transform(features)

    
    if st.button(" Predict Price"):
        pred = model.predict(features)
        price_lakhs = np.expm1(pred[0])

        if price_lakhs >= 100:
            price_cr = price_lakhs / 100
            st.success(f" Estimated Price: ₹ {price_cr:.2f} Cr")
        else:
            st.success(f" Estimated Price: ₹ {price_lakhs:.2f} Lakhs")

    st.markdown('</div>', unsafe_allow_html=True)